# HI-VIS — Automated PPE Compliance Detection

**Le Wagon Data Science & AI · Batch #2303 · Final Project**

Point a model at a construction-site photograph and get back who is in it, what PPE they're wearing, and what's missing — turning site photos every project already takes into a checked, dated, searchable safety record.

This notebook is the shared starting point: data loading, EDA, a CNN baseline, and a YOLO object-detection model.


## 1 · Setup

In [ ]:
# Core
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Modelling (transfer learning baseline)
# import tensorflow as tf
# from tensorflow.keras import layers, Sequential
# from tensorflow.keras.applications.vgg16 import VGG16

# Object detection
# from ultralytics import YOLO   # pip install ultralytics

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## 2 · Data

**Primary dataset:** [Construction Site Safety Image Dataset (Roboflow export)](https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow) — 2,801 images, YOLOv8 format, CC BY 4.0.

Classes (10): `Hardhat`, `Mask`, `NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`, `Person`, `Safety Cone`, `Safety Vest`, `machinery`, `vehicle`.

Reference notebook using this same dataset: [PPE Kit Detection — Construction Site Safety](https://www.kaggle.com/code/sajjadalishah/ppe-kit-detection-construction-site-safety).



### ⚠️ If you downloaded the reference notebook's "Output" from Kaggle

Kaggle lets you download a public notebook's entire output bundle, not just the input dataset — put in all in `data/`, it unpacks into **three unrelated things**, not one clean dataset:

```
data/
├── css-data/                          ← the ACTUAL dataset — use this as DATA_DIR
│   ├── README.dataset.txt / README.roboflow.txt
│   ├── train/{images,labels}/
│   ├── valid/{images,labels}/
│   └── test/{images,labels}/
├── results_yolov8n_100e/              ← the reference notebook's OWN completed training run
│   └── kaggle/working/
│       ├── __notebook__.ipynb         ← the reference notebook's actual source + outputs
│       ├── yolov8n.pt                 ← base COCO checkpoint (untouched)
│       ├── ppe_data.yaml              ← the data.yaml that run was trained with
│       └── runs/detect/train/weights/
│           ├── best.pt                ← already fine-tuned on THIS dataset (100 epochs)
│           └── last.pt
└── source_files/source_files/         ← random demo images/videos, not training data
```


In [2]:
#DATA_DIR = Path("data/css-data")
DATA_DIR = Path("C:/P0-safety-data/Altec PPE")

for split in ["train", "valid", "test"]:
    img_dir = DATA_DIR / split / "images"
    lbl_dir = DATA_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
    print(f"{split:6s} images={n_img:5d}  labels={n_lbl:5d}")


train  images=16209  labels=16209
valid  images= 2027  labels= 2027
test   images= 2024  labels= 2024


In [ ]:
# The Roboflow export doesn't always ship its own data.yaml (ours didn't) — write one.
# Class order below matches ppe_data.yaml from the reference notebook's run, so it's
# consistent with results_yolov8n_100e's weights too.
import yaml


# TODO: fix the path to point it at your downloaded data folder

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

data_cfg = {
    "train": str((DATA_DIR / "train" / "images").resolve()),
    "val": str((DATA_DIR / "valid" / "images").resolve()),
    "test": str((DATA_DIR / "test" / "images").resolve()),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATA_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(f"wrote {yaml_path}")
print(data_cfg)


In [3]:
#FROM Altect YAML FILE

CLASS_NAMES = ['Face_masks', 'Face_shield', 'Glasses', 'Gloves', 'Helmet', 'Safety_shoes', 'Safety_vests', 'glasses', 'helmet']

data_cfg = {
    "train": str((DATA_DIR / "train" / "images").resolve()),
    "val": str((DATA_DIR / "valid" / "images").resolve()),
    "test": str((DATA_DIR / "test" / "images").resolve()),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATA_DIR / "data.yaml"

### Quick sanity check — smoke-test YOLO26s on the RTX 5080

No one has fine-tuned YOLO26s on this dataset yet — unlike the YOLOv8n
checkpoint bundled with the reference Kaggle notebook, there's no
already-trained `best.pt` to sanity-check against. So instead we fine-tune
YOLO26s directly, starting with a **3-epoch smoke test** right here: enough to
confirm the data.yaml, classes and GPU are all wired correctly, not enough to
produce a usable model.

The real 100-epoch run belongs in a terminal, not a notebook cell — see the
next section.


In [ ]:
from ultralytics import YOLO

# Ultralytics resolves a *relative* project path against its own global
# SETTINGS["runs_dir"] (e.g. /ultralytics/runs), not the notebook's cwd — pass
# an absolute path so the run actually lands under notebooks/runs/.
RUNS_DIR = Path("runs/detect").resolve()

smoke_model = YOLO("yolo26s.pt")
smoke_results = smoke_model.train(
    data=str(yaml_path),
    epochs=3,          # smoke test only — proves the pipeline runs, not a usable model
    imgsz=640,
    batch=20,           # autobatch, sized to the RTX 5080 rather than a guessed constant
    device=0,           # RTX 5080
    project=str(RUNS_DIR),
    name="yolo26s_Altec_PPE_smoke",
    exist_ok=True,
)

smoke_best = smoke_results.save_dir / "weights" / "best.pt"
sanity_model = YOLO(str(smoke_best))
sample_image = random.choice(list((DATA_DIR / "test" / "images").glob("*")))
results = sanity_model(str(sample_image))
results[0].show()   # or results[0].save("sanity_check.jpg")


### Person + PPE association

`ppe_model` is fine-tuned on Altec PPE, which has no `person` class — see
`data/Altec PPE/data.yaml`. `ppe_association.assess()` needs a person box to
anchor its head/torso zones, so it can't run on `ppe_model`'s output alone.

Pair it with a fresh COCO-pretrained detector (`yolo26s.pt`, unmodified — not
`smoke_model`, which `.train()` already overwrote in place with the 9-class
PPE head) for the person boxes, then merge both models' detections with
`from_ultralytics_multi` before scoring compliance.

In [ ]:
from ultralytics import YOLO
from ppe_association import assess, from_ultralytics_multi

train_best = Path(r'J:\Programming\P0-safety\runs\detect\yolo26s_Altec_PPE_100e\weights') / "best.pt"
sample_image = random.choice(list((DATA_DIR / "test" / "images").glob("*")))

person_model = YOLO("yolo26s.pt")        # fresh COCO checkpoint — still has "person" (class 0)
pose_model = YOLO("yolo26s-pose.pt")     # COCO-pose checkpoint — different backbone/training,
                                          # catches some people the detection head misses
ppe_model = YOLO(str(train_best))        # fine-tuned smoke-test weights — PPE items only

person_result = person_model(str(sample_image), classes=[0], conf=0.40, verbose=False)[0]
pose_result = pose_model(str(sample_image), conf=0.40, verbose=False)[0]
ppe_result = ppe_model(str(sample_image), conf=0.35, verbose=False)[0]

# dedupe_iou merges the two person sources instead of double-counting anyone
# both models agree on
detections = from_ultralytics_multi(person_result, pose_result, ppe_result, dedupe_iou=0.5)
h, w = person_result.orig_shape  # Ultralytics gives (height, width)
assessments = assess(detections, frame_size=(w, h))

for a in assessments:
    p = a.person
    print(
        f"person @({p.x1:.0f},{p.y1:.0f},{p.x2:.0f},{p.y2:.0f}) "
        f"-> {a.overall.value:<14} {a.explain()}"
    )


In [ ]:
annotated = person_result.plot()            # draws person boxes (BGR array)
annotated = pose_result.plot(img=annotated) # layers pose-model person boxes + skeletons on top
annotated = ppe_result.plot(img=annotated)  # layers PPE boxes on top of the same image

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


### The real training run — from a terminal, not a notebook cell

100 epochs matches the reference YOLOv8n run, so the results are comparable.
Run it detached so it survives a closed tab or a restarted kernel — training
for hours inside a notebook cell risks losing the whole run to a dropped
websocket, same reasoning as `src/train_ppe.py` elsewhere in this repo.

`project` is given as an absolute path for the same reason as the smoke-test
cell above — Ultralytics rebases a relative one against its own global
`runs_dir`, not your cwd.

```bash
cd /workspace/notebooks

nohup yolo detect train \
    data=data/css-data/data.yaml model=yolo26s.pt \
    epochs=100 imgsz=640 batch=-1 device=0 patience=25 \
    project=/workspace/notebooks/runs/detect name=yolo26s_css_100e \
    > runs/train.log 2>&1 &

tail -f runs/train.log
```

Best weights land at `/workspace/notebooks/runs/detect/yolo26s_css_100e/weights/best.pt`.


## 3 · Exploratory Data Analysis

Same instinct as `Module 22`: look at the structure of the data *before* trusting any model output. For a detection dataset that means, at minimum:

1. **Class balance** — per the pitch's own risk register, `NO-Hardhat` / `NO-Mask` / `NO-Safety Vest` are expected to be far rarer than their positive counterparts. Confirm it, don't assume it.
2. **Image properties** — resolution, aspect ratio, lighting variety (site photos won't be as clean as ImageNet).
3. **Box size distribution** — small/occluded objects (a bare head at a distance) are called out in the pitch as the hardest case; know how many of those you actually have.
4. **Visual sanity check** — draw the YOLO-format label boxes on a handful of images to catch annotation issues early.


In [ ]:
def parse_yolo_labels(label_path, class_names):
    """Read a YOLO-format .txt label file into a list of (class_name, x, y, w, h) — all normalised 0-1."""
    rows = []
    if not Path(label_path).exists():
        return rows
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            rows.append((class_names[cls_id], x, y, w, h))
    return rows


In [ ]:
# Class distribution across the training split
records = []
train_labels_dir = DATA_DIR / "train" / "labels"
if train_labels_dir.exists():
    for lbl_file in train_labels_dir.glob("*.txt"):
        for cls_name, x, y, w, h in parse_yolo_labels(lbl_file, CLASS_NAMES):
            records.append({"class": cls_name, "w": w, "h": h, "area": w * h})

df_labels = pd.DataFrame(records)

if not df_labels.empty:
    plt.figure(figsize=(9, 4))
    order = df_labels["class"].value_counts().index
    sns.countplot(data=df_labels, y="class", order=order)
    plt.title("Bounding box count per class (train split)")
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()
else:
    print("No labels parsed yet — download the dataset first.")


In [ ]:
# Box-area distribution (proxy for "how many small/occluded objects are we dealing with?")
if not df_labels.empty:
    plt.figure(figsize=(9, 4))
    sns.histplot(df_labels["area"], bins=40)
    plt.title("Bounding box area (normalised) distribution")
    plt.xlabel("box area (fraction of image)")
    plt.show()

    print(df_labels.groupby("class")["area"].median().sort_values())


In [ ]:
def draw_boxes(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    for cls_name, x, y, w, h in parse_yolo_labels(label_path, class_names):
        x1, y1 = int((x - w / 2) * w_img), int((y - h / 2) * h_img)
        x2, y2 = int((x + w / 2) * w_img), int((y + h / 2) * h_img)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, cls_name, (x1, max(y1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return img


# Sanity-check a handful of random training images with their labels overlaid
train_images_dir = DATA_DIR / "train" / "images"
if train_images_dir.exists():
    sample_images = random.sample(list(train_images_dir.glob("*")), k=min(6, len(list(train_images_dir.glob("*")))))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample_images):
        lbl_path = train_labels_dir / (img_path.stem + ".txt")
        ax.imshow(draw_boxes(img_path, lbl_path, CLASS_NAMES))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 4 · Basic CNN

Use transfer learning and build a basic classifier to establish a floor.
We test with ImageNet which is a huge public dataset (about 1.4 million photos across 1,000 everyday categories (dogs, cars, chairs, bottles, and so on)). 

The hardhat/no-hardhat distinction comes from the dataset's labels  (the YOLO `.txt`).

1) read each label file, 
2) keep only the `Hardhat` / `NO-Hardhat` boxes
3) crop just that region out of the source image (in case there are multiple boxes in a photo that matters for training)
4) save each crop into `data/baseline_crops/hardhat/` or `data/baseline_crops/no_hardhat/`. 

`tf.keras.utils.image_dataset_from_directory` infers the label for every image purely from which of those two folders it's sitting in.


In [ ]:
def crop_head_regions(images_dir, labels_dir, class_names, output_dir, padding=0.1):
    """Crop the Hardhat / NO-Hardhat boxes out of each image and sort the crops into
    output_dir/hardhat/ and output_dir/no_hardhat/ — the folder split
    image_dataset_from_directory (next cell) uses to infer labels automatically."""
    output_dir = Path(output_dir)
    (output_dir / "hardhat").mkdir(parents=True, exist_ok=True)
    (output_dir / "no_hardhat").mkdir(parents=True, exist_ok=True)
    n_saved = {"hardhat": 0, "no_hardhat": 0}

    for label_path in Path(labels_dir).glob("*.txt"):
        image_path = Path(images_dir) / (label_path.stem + ".jpg")
        if not image_path.exists():
            continue
        img = cv2.imread(str(image_path))
        if img is None:
            continue
        h_img, w_img = img.shape[:2]

        for i, (cls_name, x, y, w, h) in enumerate(parse_yolo_labels(label_path, class_names)):
            if cls_name not in ("Hardhat", "NO-Hardhat"):
                continue

            # normalized YOLO box -> pixel box, with a little padding around the head
            bw, bh = w * (1 + padding), h * (1 + padding)
            x1 = max(int((x - bw / 2) * w_img), 0)
            y1 = max(int((y - bh / 2) * h_img), 0)
            x2 = min(int((x + bw / 2) * w_img), w_img)
            y2 = min(int((y + bh / 2) * h_img), h_img)

            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            folder = "hardhat" if cls_name == "Hardhat" else "no_hardhat"
            cv2.imwrite(str(output_dir / folder / f"{label_path.stem}_{i}.jpg"), crop)
            n_saved[folder] += 1

    return n_saved


crop_counts = crop_head_regions(
    images_dir=DATA_DIR / "train" / "images",
    labels_dir=DATA_DIR / "train" / "labels",
    class_names=CLASS_NAMES,
    output_dir=DATA_DIR / "baseline_crops",
)
print(crop_counts)

In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "baseline_crops",
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary",     # single 0/1 label — matches the sigmoid output below
    validation_split=0.2,
    subset="training",
    seed=RANDOM_STATE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "baseline_crops",
    image_size=(224, 224),
    batch_size=32,
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=RANDOM_STATE,
)

print(train_ds.class_names)  # expect: ['hardhat', 'no_hardhat'] -> hardhat=0, no_hardhat=1


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras import layers, Sequential

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # freeze the convolutional base, same as the bootcamp challenge

baseline_model = Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),  # hardhat vs no-hardhat
])
baseline_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping],
)

In [ ]:
loss, accuracy = baseline_model.evaluate(val_ds)
print(f"val loss: {loss:.3f}, val accuracy: {accuracy:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.show()